# `master_genre.csv` — Exploratory Data Analysis

Explore Last.fm enrichment coverage, all 48 columns, track vs. artist genre breakdowns, top genres/tags, similar tracks/artists, and track popularity metrics across `data/master_genre.csv`.

## 1. Setup, Path Resolution & Dataset Loading

In [ ]:
import csv
from collections import Counter
from pathlib import Path

try:
    import pandas as pd

    pd.set_option("display.max_columns", None)
    pd.set_option("display.max_colwidth", 80)
    pd.set_option("display.width", 200)
    HAS_PANDAS = True
except ImportError:
    pd = None
    HAS_PANDAS = False

try:
    from IPython.display import display
except ImportError:
    display = print


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "master_genre.csv").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    return start


PROJECT_DIR = find_project_root(Path.cwd().resolve())
MASTER_CSV = PROJECT_DIR / "data" / "master.csv"
GENRE_CSV = PROJECT_DIR / "data" / "master_genre.csv"

with GENRE_CSV.open(encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))
    columns = list(rows[0].keys()) if rows else []

master_count = 0
if MASTER_CSV.exists():
    with MASTER_CSV.open(encoding="utf-8-sig", newline="") as f:
        master_count = sum(1 for _ in csv.DictReader(f))

df = pd.DataFrame(rows) if HAS_PANDAS else None


def to_int(value: str, default: int = 0) -> int:
    value = (value or "").strip()
    try:
        return int(value)
    except ValueError:
        return default


def show_table(records: list[dict], title: str = "", max_rows: int = 25) -> None:
    if title:
        print(f"\n=== {title} (showing {min(len(records), max_rows)} of {len(records):,}) ===")
    if not records:
        print("(no rows)")
        return
    if HAS_PANDAS:
        display(pd.DataFrame(records[:max_rows]))
    else:
        cols = list(records[0].keys())
        header = " | ".join(cols)
        print(header)
        print("-" * min(120, len(header) + 20))
        for rec in records[:max_rows]:
            print(" | ".join(str(rec.get(c, ""))[:60] for c in cols))


pct_done = (len(rows) / master_count * 100) if master_count else 0.0
print(f"Project root:      {PROJECT_DIR}")
print(f"Enriched CSV:      {GENRE_CSV} ({len(rows):,} rows, {len(columns)} columns)")
print(f"Master CSV total:  {master_count:,} rows ({pct_done:.2f}% enriched)")
print(f"pandas available:  {HAS_PANDAS}")

## 2. All 48 Columns, Fill Rates & Sample Rows

In [ ]:
print(f"All {len(columns)} columns in master_genre.csv:")
for i, col in enumerate(columns, start=1):
    print(f"  {i:2d}. {col}")

fill_stats = []
total_rows = len(rows)
for col in columns:
    non_empty = sum(1 for r in rows if (r.get(col) or "").strip() != "")
    pct = (non_empty / total_rows * 100) if total_rows else 0.0
    fill_stats.append({
        "column": col,
        "non_empty": non_empty,
        "missing": total_rows - non_empty,
        "fill_pct": round(pct, 2),
    })

show_table(fill_stats, title="Column Fill Rates", max_rows=len(columns))
show_table(rows[:5], title="Sample Rows (All 48 Columns)", max_rows=5)

## 3. Genre Coverage Breakdown (Track Genre vs. Artist Genre vs. None)

In [ ]:
source_counts = Counter((r.get("genre_source") or "none").strip() for r in rows)
has_any_genre = sum(1 for r in rows if (r.get("genres_all") or "").strip())
has_track_genre = sum(1 for r in rows if to_int(r.get("track_tag_count", "0")) > 0)
has_artist_genre = sum(1 for r in rows if to_int(r.get("artist_tag_count", "0")) > 0)
has_both = sum(
    1
    for r in rows
    if to_int(r.get("track_tag_count", "0")) > 0 and to_int(r.get("artist_tag_count", "0")) > 0
)

n = max(1, len(rows))
coverage_summary = [
    {"metric": "Songs with any genre (genres_all non-empty)", "count": has_any_genre, "pct": round(has_any_genre / n * 100, 2)},
    {"metric": "Songs with track-level genre (track_tag_count > 0)", "count": has_track_genre, "pct": round(has_track_genre / n * 100, 2)},
    {"metric": "Songs with artist-level genre (artist_tag_count > 0)", "count": has_artist_genre, "pct": round(has_artist_genre / n * 100, 2)},
    {"metric": "Songs with both track & artist tags", "count": has_both, "pct": round(has_both / n * 100, 2)},
    {"metric": "Songs with no genre (genre_source == 'none')", "count": source_counts.get("none", 0), "pct": round(source_counts.get("none", 0) / n * 100, 2)},
]
show_table(coverage_summary, title="Genre Coverage Summary", max_rows=10)

source_breakdown = [
    {"genre_source": src, "songs": cnt, "pct": round(cnt / n * 100, 2)}
    for src, cnt in source_counts.most_common()
]
show_table(source_breakdown, title="Breakdown by genre_source", max_rows=10)

preview_cols = ["rank", "artist_names", "track_name", "primary_artist", "genre_source", "track_tag_count", "artist_tag_count", "genres_all"]
for src in ("track+artist", "track", "artist", "none"):
    subset = [{c: r.get(c, "") for c in preview_cols} for r in rows if (r.get("genre_source") or "none").strip() == src]
    if subset:
        show_table(subset, title=f"Sample rows for genre_source='{src}'", max_rows=5)

## 4. Top Genres & Tag Distributions

In [ ]:
def parse_weighted_tags(raw: str) -> list[tuple[str, int]]:
    parsed = []
    for item in (raw or "").split("|"):
        item = item.strip()
        if not item:
            continue
        if item.endswith(")") and "(" in item:
            name_part, weight_part = item.rsplit("(", 1)
            parsed.append((name_part.strip().casefold(), to_int(weight_part[:-1], 0)))
        else:
            parsed.append((item.casefold(), 0))
    return parsed


combined_song_freq = Counter()
track_tag_freq = Counter()
artist_tag_freq = Counter()
track_weighted_score = Counter()
artist_weighted_score = Counter()

for r in rows:
    for tag in {t.strip().casefold() for t in (r.get("genres_all") or "").split("|") if t.strip()}:
        combined_song_freq[tag] += 1
    for tag, weight in parse_weighted_tags(r.get("track_tags_all", "")):
        track_tag_freq[tag] += 1
        track_weighted_score[tag] += weight
    for tag, weight in parse_weighted_tags(r.get("artist_tags_all", "")):
        artist_tag_freq[tag] += 1
        artist_weighted_score[tag] += weight

t_counts = [to_int(r.get("track_tag_count", "0")) for r in rows]
a_counts = [to_int(r.get("artist_tag_count", "0")) for r in rows]
t_nonzero = [c for c in t_counts if c > 0]
a_nonzero = [c for c in a_counts if c > 0]

print(f"Unique combined tags: {len(combined_song_freq):,}")
print(f"Unique track tags:    {len(track_tag_freq):,} (avg per matched song: {sum(t_nonzero)/max(1, len(t_nonzero)):.2f}, max: {max(t_counts, default=0)})")
print(f"Unique artist tags:   {len(artist_tag_freq):,} (avg per matched song: {sum(a_nonzero)/max(1, len(a_nonzero)):.2f}, max: {max(a_counts, default=0)})")

top_combined = [
    {"rank": i, "tag": tag, "song_count": cnt, "pct_of_songs": round(cnt / n * 100, 2)}
    for i, (tag, cnt) in enumerate(combined_song_freq.most_common(30), start=1)
]
top_track = [
    {"rank": i, "tag": tag, "song_count": cnt, "total_weight": track_weighted_score[tag]}
    for i, (tag, cnt) in enumerate(track_tag_freq.most_common(25), start=1)
]
top_artist = [
    {"rank": i, "tag": tag, "song_count": cnt, "total_weight": artist_weighted_score[tag]}
    for i, (tag, cnt) in enumerate(artist_tag_freq.most_common(25), start=1)
]

show_table(top_combined, title="Top 30 Combined Tags (genres_all)", max_rows=30)
show_table(top_track, title="Top 25 Track-Level Tags (track_tags_names / weights)", max_rows=25)
show_table(top_artist, title="Top 25 Artist-Level Tags (artist_tags_names / weights)", max_rows=25)

## 5. Songs with Similar Tracks & Similar Artists

In [ ]:
sim_track_rows = [
    {
        "rank": r.get("rank", ""),
        "artist_names": r.get("artist_names", ""),
        "track_name": r.get("track_name", ""),
        "genre_source": r.get("genre_source", ""),
        "genres_all": r.get("genres_all", ""),
        "similar_tracks": r.get("similar_tracks", ""),
    }
    for r in rows
    if (r.get("similar_tracks") or "").strip()
]

sim_artist_rows_count = sum(1 for r in rows if (r.get("similar_artists") or "").strip())
sim_artist_freq = Counter()
for r in rows:
    for name in (r.get("similar_artists") or "").split("|"):
        clean = name.strip()
        if clean:
            sim_artist_freq[clean] += 1

print(f"Songs with similar_tracks populated:  {len(sim_track_rows):,} / {len(rows):,} ({len(sim_track_rows)/n*100:.2f}%)")
print(f"Songs with similar_artists populated: {sim_artist_rows_count:,} / {len(rows):,} ({sim_artist_rows_count/n*100:.2f}%)")

show_table(sim_track_rows, title="Songs with Similar Tracks", max_rows=30)

top_sim_artists = [
    {"rank": i, "similar_artist": artist, "referenced_in_songs": cnt}
    for i, (artist, cnt) in enumerate(sim_artist_freq.most_common(20), start=1)
]
show_table(top_sim_artists, title="Top 20 Most Frequently Referenced Similar Artists", max_rows=20)

## 6. Additional Metadata & Popularity Analytics

In [ ]:
mbid_track = sum(1 for r in rows if (r.get("lfm_track_mbid") or "").strip())
mbid_artist = sum(1 for r in rows if (r.get("lfm_artist_mbid") or "").strip())
dur_positive = sum(1 for r in rows if to_int(r.get("duration_ms", "0")) > 0)
album_present = sum(1 for r in rows if (r.get("album_title") or "").strip())
wiki_present = sum(1 for r in rows if (r.get("track_wiki_summary") or "").strip())
bio_present = sum(1 for r in rows if (r.get("artist_bio_summary") or "").strip())

metadata_summary = [
    {"field": "lfm_track_mbid", "count": mbid_track, "pct": round(mbid_track / n * 100, 2)},
    {"field": "lfm_artist_mbid", "count": mbid_artist, "pct": round(mbid_artist / n * 100, 2)},
    {"field": "duration_ms > 0", "count": dur_positive, "pct": round(dur_positive / n * 100, 2)},
    {"field": "album_title", "count": album_present, "pct": round(album_present / n * 100, 2)},
    {"field": "track_wiki_summary", "count": wiki_present, "pct": round(wiki_present / n * 100, 2)},
    {"field": "artist_bio_summary", "count": bio_present, "pct": round(bio_present / n * 100, 2)},
]
show_table(metadata_summary, title="Metadata Coverage Analytics", max_rows=10)

artist_track_counts = Counter((r.get("primary_artist") or "").strip() for r in rows if (r.get("primary_artist") or "").strip())
top_artists = [
    {"rank": i, "primary_artist": artist, "tracks_in_master_genre": cnt}
    for i, (artist, cnt) in enumerate(artist_track_counts.most_common(20), start=1)
]
show_table(top_artists, title="Top 20 Primary Artists by Track Count", max_rows=20)

pop_cols = ["rank", "primary_artist", "track_name", "streams", "track_listeners", "track_playcount", "genre_source"]
top_by_streams = [
    {c: r.get(c, "") for c in pop_cols}
    for r in sorted(rows, key=lambda r: to_int(r.get("streams", "0")), reverse=True)[:15]
]
top_by_lfm_playcount = [
    {c: r.get(c, "") for c in pop_cols}
    for r in sorted(rows, key=lambda r: to_int(r.get("track_playcount", "0")), reverse=True)[:15]
]
show_table(top_by_streams, title="Top 15 Tracks by Spotify Chart Streams", max_rows=15)
show_table(top_by_lfm_playcount, title="Top 15 Tracks by Last.fm Playcount", max_rows=15)